# RL Training Notebook v3 (Google Colab) — Phase 07 Training v3

Run **top to bottom** after **Runtime → Factory reset**. Artifacts write to:
`/content/drive/MyDrive/evaluator-gym-phase07-v3-run2`.

If SFT + preflight already finished on Drive, use the **RL-only resume** cell instead of re-running base/SFT/preflight.

**Memory (T4):** smoke reuses the SFT model in-place (no second `build_policy()`). Beta runs reload once after smoke is released.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

assert os.path.exists('/content'), 'Run this notebook in Google Colab'
subprocess.run(['nvidia-smi'], check=True)

REPO_URL = 'https://github.com/prashere/evaluator_gym.git'
COLAB_BRANCH = 'rl_v3'

try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = os.environ.get('GITHUB_TOKEN')
if github_token:
    REPO_URL = f'https://{github_token}@github.com/prashere/evaluator_gym.git'

REPO = Path('/content/evaluator_gym')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-b', COLAB_BRANCH, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin', COLAB_BRANCH], check=True)
subprocess.run(['git', 'checkout', COLAB_BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', COLAB_BRANCH], check=True)

_bootstrap_file = REPO / 'src/evaluator_gym/training/colab_bootstrap.py'
_spec = importlib.util.spec_from_file_location('colab_bootstrap', _bootstrap_file)
_bootstrap = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_bootstrap)
SRC = _bootstrap.install_repo_src(REPO)
print('Added to sys.path:', SRC)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO)], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'transformers==4.57.6', 'peft==0.17.1', 'bitsandbytes==0.47.0',
    'accelerate==1.10.1', 'mlflow==3.10.0', 'matplotlib>=3.8,<4', 'tqdm>=4.66,<5',
], check=True)

_bootstrap.install_repo_src(REPO)
print('Added to sys.path (post-pip):', _bootstrap.install_repo_src(REPO))
print('Phase 07 v3 imports:', _bootstrap.verify_imports_v3())

from google.colab import drive
drive.mount('/content/drive')

from evaluator_gym.training.phase07_core import ensure_output_root
from evaluator_gym.training.phase07v3_core import (
    DEFAULT_OUTPUT_ROOT_V3,
    PREVIOUS_OUTPUT_ROOT_V3,
    RESULTS_STAGING_ROOT_V3,
)
from evaluator_gym.training.phase07v3_notebook import verify_bitsandbytes, write_run_manifest

OUTPUT_ROOT = ensure_output_root(DEFAULT_OUTPUT_ROOT_V3)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('bitsandbytes:', verify_bitsandbytes())
RUN_MANIFEST = write_run_manifest(
    OUTPUT_ROOT,
    repo_commit=COMMIT,
    colab_branch=COLAB_BRANCH,
    extra={
        'previous_output_root': str(PREVIOUS_OUTPUT_ROOT_V3),
        'results_staging_root': str(RESULTS_STAGING_ROOT_V3),
        'run_mode': 'full_top_to_bottom',
    },
)
print('run_manifest:', RUN_MANIFEST)
print('Commit:', COMMIT)


In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

from evaluator_gym.training.phase07v3_core import build_phase07v3_splits

TRAIN_CONFIG, HELDOUT_POOL_CONFIG, TRAIN_TASK_ROWS, HELDOUT_TASK_ROWS = build_phase07v3_splits()
TRAIN_TASKS = [row.as_dict() for row in TRAIN_TASK_ROWS]
HELDOUT_TASKS = [row.as_dict() for row in HELDOUT_TASK_ROWS]

print('Train:', TRAIN_CONFIG.to_dict())
print('Held-out pool:', HELDOUT_POOL_CONFIG.to_dict())
print('Tier counts:', dict(Counter(row['tier'] for row in TRAIN_TASKS)))
print('Held-out IDs:', [row['task_id'] for row in HELDOUT_TASKS])


In [ ]:
import sys
from pathlib import Path

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

from evaluator_gym.training.phase07v3_core import build_training_schedule_v3, TARGET_OPTIMIZER_STEPS
from evaluator_gym.training.phase07v3_notebook import load_v3_drive_state

DRIVE_STATE = load_v3_drive_state(OUTPUT_ROOT)
RESUME_RL_ONLY = DRIVE_STATE.get('ready_for_rl', False)
print('Drive RL resume ready:', RESUME_RL_ONLY)
if RESUME_RL_ONLY:
    GATE = DRIVE_STATE['gate']
    PREFLIGHT = DRIVE_STATE['preflight']
    BASE_HELDOUT = DRIVE_STATE.get('base_heldout', {})
    POST_SFT_HELDOUT = DRIVE_STATE.get('post_sft_heldout', {})
    TRAINABLE_IDS = set(DRIVE_STATE['trainable_task_ids'])
    TRAINING_SCHEDULE = build_training_schedule_v3(TRAINABLE_IDS, TRAIN_TASK_ROWS, TARGET_OPTIMIZER_STEPS)
    print('Loaded gate + preflight from Drive; skip base/SFT/preflight cells and run smoke onward.')
else:
    print('No complete Drive state — run base, SFT, and preflight cells below.')



In [ ]:
import random
import sys
from pathlib import Path

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

import mlflow
import numpy as np
import torch
from evaluator_gym.rubric import RUBRIC_VERSION
from evaluator_gym.training_rubric.binary import TRAINING_RUBRIC_VERSION_BINARY
from evaluator_gym.training.phase07_core import (
    MAX_COMPLETION_TOKENS,
    MAX_RETRIES,
    RUN_SEED,
    build_training_metric,
    ensure_output_root,
)
from evaluator_gym.training.phase07v3_core import (
    BETAS,
    DEFAULT_OUTPUT_ROOT_V3,
    GROUP_SIZE,
    HELDOUT_ROLLOUTS,
    MAX_RESAMPLE_ATTEMPTS,
    MAX_TOTAL_COMPLETIONS,
    MODEL_ID,
    MODEL_REVISION,
    PREFLIGHT_PROBES_PER_TASK,
    SMOKE_STEPS,
    TARGET_OPTIMIZER_STEPS,
    budget_status,
    build_training_schedule_v3,
    checkpoint_mode_v3,
    compute_rloo_advantages,
    evaluate_sft_gate,
    summarize_dual_evaluation_v3,
    summarize_preflight_v3,
    validate_trainable_pool_v3,
)
from evaluator_gym.training.phase07v3_notebook import (
    backward_rloo_policy_step,
    load_sft_adapter_weights,
    release_gpu_memory,
    sft_adapter_dir,
    sft_adapter_ready,
)
from evaluator_gym.training.phase07v3_runtime import (
    append_jsonl,
    exploit_search_v3,
    read_jsonl,
    run_async,
    score_text_dual_v3,
)
from evaluator_gym.training.phase07_live import LiveRunLogger, tqdm_progress
from evaluator_gym.training.sft_reference import build_sft_examples

OUTPUT_ROOT = ensure_output_root(globals().get('OUTPUT_ROOT', DEFAULT_OUTPUT_ROOT_V3))

assert RUBRIC_VERSION == '0.1.2', f'Expected eval rubric 0.1.2, got {RUBRIC_VERSION}'
assert TRAINING_RUBRIC_VERSION_BINARY == 'train-0.2.0'
assert torch.cuda.is_available(), 'A Colab GPU is required'
print('GPU:', torch.cuda.get_device_name(0))
print('Model:', MODEL_ID)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('GROUP_SIZE:', GROUP_SIZE, '| budget cap:', MAX_TOTAL_COMPLETIONS)
mlflow.set_tracking_uri((OUTPUT_ROOT / 'mlruns').as_uri())
mlflow.set_experiment('phase07-training-v3')
print('SFT adapter path:', sft_adapter_dir(OUTPUT_ROOT))
print('MLflow tracking:', OUTPUT_ROOT / 'mlruns')


In [ ]:
import sys
from pathlib import Path

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed


def build_policy():
    set_seed(RUN_SEED)
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        quantization_config=quantization,
        device_map={'': 0},
        dtype=torch.float16,
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    ))
    unexpected = [name for name, p in model.named_parameters() if p.requires_grad and 'lora_' not in name]
    assert not unexpected, f'Frozen-reference invariant failed: {unexpected[:5]}'
    return model, tokenizer


def render_prompt(task, tokenizer):
    text = tokenizer.apply_chat_template(task['prompt'], tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(text, return_tensors='pt').to('cuda')
    assert encoded['input_ids'].shape[1] + MAX_COMPLETION_TOKENS <= 32768
    return encoded


def token_statistics(model, sample):
    tokens = torch.cat([sample['prompt_ids'], sample['completion_ids']]).unsqueeze(0)
    targets = sample['completion_ids']
    keep = targets.numel() + 1
    policy_logits = model(tokens, logits_to_keep=keep).logits[0, :-1].float()
    policy_log_probs = policy_logits.log_softmax(-1)
    policy_token_logp = policy_log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    entropy = -(policy_log_probs.exp() * policy_log_probs).sum(-1).mean()
    with torch.no_grad(), model.disable_adapter():
        reference_logits = model(tokens, logits_to_keep=keep).logits[0, :-1].float()
        reference_token_logp = reference_logits.log_softmax(-1).gather(1, targets.unsqueeze(1)).squeeze(1)
    log_ratio = reference_token_logp - policy_token_logp
    k3 = (torch.exp(log_ratio) - log_ratio - 1).mean()
    return policy_token_logp.mean(), k3, entropy


In [ ]:
import sys
from pathlib import Path

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

from peft import get_peft_model_state_dict, set_peft_model_state_dict


def save_checkpoint(run_dir, tag, model, optimizer=None, optimizer_applied_steps=0, *, mode='adapter'):
    if mode == 'full':
        checkpoint = run_dir / tag
        checkpoint.mkdir(parents=True, exist_ok=True)
        torch.save({
            'optimizer_applied_steps': optimizer_applied_steps,
            'adapter': get_peft_model_state_dict(model),
            'optimizer': optimizer.state_dict() if optimizer is not None else None,
            'python_rng': random.getstate(),
            'numpy_rng': np.random.get_state(),
            'torch_rng': torch.get_rng_state(),
            'cuda_rng': torch.cuda.get_rng_state_all(),
        }, checkpoint / 'state.pt')
        return checkpoint
    checkpoint = run_dir / tag
    model.save_pretrained(checkpoint)
    return checkpoint


def evaluate_policy(model, tokenizer, run_name):
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    transcript_path = run_dir / 'heldout_rollouts.jsonl'
    if transcript_path.exists():
        transcript_path.unlink()
    live = LiveRunLogger(run_name, run_dir)
    live.print_banner(f'Held-out evaluation — {run_name}')
    model.eval()
    total = len(HELDOUT_TASKS) * HELDOUT_ROLLOUTS
    rollout_index_global = 0
    progress = tqdm_progress(range(total), desc=f'held-out {run_name}', total=total)
    for task_index, task in enumerate(HELDOUT_TASKS):
        encoded = render_prompt(task, tokenizer)
        prompt_length = encoded['input_ids'].shape[1]
        for rollout_index in range(HELDOUT_ROLLOUTS):
            seed = RUN_SEED + task_index * HELDOUT_ROLLOUTS + rollout_index
            torch.manual_seed(seed)
            with torch.no_grad():
                output = model.generate(
                    **encoded, do_sample=True, temperature=0.8,
                    max_new_tokens=MAX_COMPLETION_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            ids = output[0, prompt_length:]
            text = tokenizer.decode(ids, skip_special_tokens=True)
            training_reward, eval_reward, parse_result, audit = run_async(score_text_dual_v3(task, text))
            rollout_index_global += 1
            append_jsonl(transcript_path, {
                'run': run_name, 'task_id': task['task_id'], 'tier': task['tier'],
                'rollout_index': rollout_index, 'seed': seed, 'completion': text,
                'completion_tokens': int(ids.numel()), 'reward': eval_reward, 'eval_reward': eval_reward,
                'training_reward': training_reward, 'parse_result': parse_result, 'reward_audit': audit,
            })
            live.log_eval_rollout(
                index=rollout_index_global, total=total, task_id=task['task_id'], tier=task['tier'],
                reward=eval_reward, completion_tokens=int(ids.numel()), parse_result=parse_result,
            )
            progress.update(1)
    progress.close()
    rows = read_jsonl(transcript_path)
    summary = summarize_dual_evaluation_v3(rows)
    (run_dir / 'heldout_summary.json').write_text(json.dumps(summary, indent=2))
    return summary


def generate_one_scored(model, tokenizer, task, seed, rejection_path=None, meta=None):
    encoded = render_prompt(task, tokenizer)
    prompt_length = encoded['input_ids'].shape[1]
    torch.manual_seed(seed)
    model.eval()
    with torch.inference_mode():
        output = model.generate(
            **encoded, do_sample=True, temperature=0.8,
            max_new_tokens=MAX_COMPLETION_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    completion_ids = output[0, prompt_length:].detach()
    text = tokenizer.decode(completion_ids, skip_special_tokens=True)
    training_reward, eval_reward, parse_result, audit = run_async(score_text_dual_v3(task, text))
    row = {
        **(meta or {}),
        'task_id': task['task_id'], 'tier': task['tier'], 'completion': text,
        'completion_tokens': int(completion_ids.numel()), 'reward': training_reward,
        'training_reward': training_reward, 'eval_reward': eval_reward,
        'parse_result': parse_result, 'reward_audit': audit,
    }
    if training_reward is None and rejection_path is not None:
        append_jsonl(rejection_path, row)
        return None
    row['prompt_ids'] = encoded['input_ids'][0].detach()
    row['completion_ids'] = completion_ids
    return row


def generate_mixed_group(model, tokenizer, task, nominal_step, rejection_path, live_logger=None, max_resample_attempts=None):
    from evaluator_gym.training.phase07v3_core import is_mixed_binary_group
    max_resample_attempts = max_resample_attempts or MAX_RESAMPLE_ATTEMPTS
    model.eval()
    for attempt in range(max_resample_attempts):
        samples = []
        for group_index in range(GROUP_SIZE):
            accepted = None
            for retry in range(MAX_RETRIES):
                seed = RUN_SEED + nominal_step * 1000 + attempt * 100 + group_index * 10 + retry
                sample = generate_one_scored(
                    model, tokenizer, task, seed, rejection_path,
                    meta={'nominal_step': nominal_step, 'group_index': group_index, 'retry': retry, 'resample_attempt': attempt + 1, 'seed': seed},
                )
                if sample is None:
                    continue
                accepted = sample
                break
            if accepted is None:
                samples = None
                break
            samples.append(accepted)
        if samples is None:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue
        rewards = [float(s['training_reward']) for s in samples]
        if is_mixed_binary_group(rewards):
            return samples, attempt + 1
    return None, max_resample_attempts


def run_preflight(model, tokenizer):
    probes = []
    live = LiveRunLogger('preflight', OUTPUT_ROOT)
    model.eval()
    for task_index, task_row in enumerate(TRAIN_TASK_ROWS):
        task = task_row.as_dict()
        for probe_index in range(PREFLIGHT_PROBES_PER_TASK):
            seed = RUN_SEED + 500000 + task_index * 100 + probe_index
            sample = generate_one_scored(model, tokenizer, task, seed)
            probes.append({
                'task_id': task['task_id'], 'tier': task['tier'], 'probe_index': probe_index,
                'reward': None if sample is None else sample['training_reward'],
                'training_reward': None if sample is None else sample['training_reward'],
                'eval_reward': None if sample is None else sample['eval_reward'],
                'parse_result': {'ok': False} if sample is None else sample['parse_result'],
                'completion_tokens': 0 if sample is None else sample['completion_tokens'],
            })
    model.eval()
    return summarize_preflight_v3(probes)


def run_sft(model, tokenizer, *, epochs=1, max_examples=100):
    examples = build_sft_examples(TRAIN_TASK_ROWS[:max_examples])
    run_dir = OUTPUT_ROOT / 'sft'
    run_dir.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=5e-6)
    model.train()
    for epoch in range(epochs):
        random.shuffle(examples)
        progress = tqdm_progress(examples, desc=f'sft epoch {epoch + 1}', total=len(examples))
        for example in progress:
            encoded = render_prompt({'prompt': example['prompt']}, tokenizer)
            completion_ids = tokenizer(example['completion'], add_special_tokens=False, return_tensors='pt')['input_ids'][0].to('cuda')
            sample = {
                'prompt_ids': encoded['input_ids'][0],
                'completion_ids': completion_ids,
            }
            policy_logp, _, _ = token_statistics(model, sample)
            loss = -policy_logp
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    save_checkpoint(run_dir, 'final-adapter', model, mode='adapter')
    tokenizer.save_pretrained(run_dir / 'final-adapter')
    return run_dir



In [ ]:
import sys
from pathlib import Path
from collections import Counter
from statistics import fmean

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

from peft import set_peft_model_state_dict


def train_run(beta, run_name, target_optimizer_steps, trainable_ids, training_schedule, resume_checkpoint=None, smoke_resume=False, policy_model=None, policy_tokenizer=None, max_resample_attempts=None):
    assert beta > 0
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = run_dir / 'metrics.jsonl'
    rollout_path = run_dir / 'training_rollouts.jsonl'
    rejection_path = run_dir / 'rejected_unscored.jsonl'
    config_path = run_dir / 'config.json'
    adapter_dir = sft_adapter_dir(OUTPUT_ROOT)
    run_config = {
        'model': MODEL_ID, 'training_rubric_version': TRAINING_RUBRIC_VERSION_BINARY, 'eval_rubric_version': RUBRIC_VERSION,
        'beta': beta, 'seed': RUN_SEED, 'target_optimizer_steps': target_optimizer_steps, 'group_size': GROUP_SIZE,
        'max_resample_attempts': MAX_RESAMPLE_ATTEMPTS, 'max_total_completions': MAX_TOTAL_COMPLETIONS,
        'trainable_task_ids': sorted(trainable_ids),
        'sft_adapter': str(adapter_dir),
        'sft_adapter_loaded': False,
    }
    if resume_checkpoint:
        if metrics_path.exists() or rollout_path.exists():
            pass
    elif metrics_path.exists() or rollout_path.exists():
        raise RuntimeError(f'{run_dir} already contains a run; delete it or resume from checkpoint')

    max_resample_attempts = max_resample_attempts or MAX_RESAMPLE_ATTEMPTS
    run_config['max_resample_attempts'] = max_resample_attempts
    hard_release_gpu_memory(
        globals(),
        'smoke_model', 'smoke_tokenizer', 'low_model', 'low_tokenizer', 'strong_model', 'strong_tokenizer',
    )
    reused_policy = policy_model is not None and policy_tokenizer is not None
    state = None
    if reused_policy:
        model, tokenizer = policy_model, policy_tokenizer
        for name in ('sft_model', 'sft_tokenizer'):
            if name in globals() and globals()[name] is model:
                del globals()[name]
        run_config['sft_adapter_loaded'] = True
        run_config['reused_policy'] = True
        log_gpu_memory(f'train_run/{run_name} reused policy')
    else:
        assert_gpu_headroom(max_allocated_gib=PRE_RL_MAX_ALLOCATED_GIB, label=run_name)
        model, tokenizer = build_policy()
        run_config['reused_policy'] = False
        if resume_checkpoint:
            state = torch.load(Path(resume_checkpoint) / 'state.pt', map_location='cpu', weights_only=False)
            set_peft_model_state_dict(model, state['adapter'])
            run_config['sft_adapter_loaded'] = True
        elif sft_adapter_ready(OUTPUT_ROOT):
            load_sft_adapter_weights(model, adapter_dir)
            run_config['sft_adapter_loaded'] = True
        else:
            raise RuntimeError(f'Missing SFT adapter at {adapter_dir}; run SFT cell first')
        log_gpu_memory(f'train_run/{run_name} loaded policy')
    if resume_checkpoint and reused_policy:
        state = torch.load(Path(resume_checkpoint) / 'state.pt', map_location='cpu', weights_only=False)
        set_peft_model_state_dict(model, state['adapter'])
        run_config['sft_adapter_loaded'] = True

    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1e-5)
    optimizer_applied_steps = 0
    total_completions = 0
    nominal_step = 0
    if resume_checkpoint and state is not None:
        if state.get('optimizer'):
            optimizer.load_state_dict(state['optimizer'])
        optimizer_applied_steps = state.get('optimizer_applied_steps', 0)
    config_path.write_text(json.dumps(run_config, indent=2))

    skip_counts = Counter()
    live = LiveRunLogger(run_name, run_dir)
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            'beta': beta,
            'group_size': GROUP_SIZE,
            'target_optimizer_steps': target_optimizer_steps,
            'sft_adapter_loaded': run_config['sft_adapter_loaded'],
        })
        while optimizer_applied_steps < target_optimizer_steps and total_completions < MAX_TOTAL_COMPLETIONS:
            nominal_step += 1
            task_row = training_schedule[(nominal_step - 1) % len(training_schedule)]
            if task_row is None:
                metric = build_training_metric(nominal_step=nominal_step, task=None, rewards=None, kl=None, entropy=None,
                    mean_completion_length=None, optimizer_applied=False, skip_reason='no_trainable_task')
                append_jsonl(metrics_path, metric)
                skip_counts['no_trainable_task'] += 1
                continue
            task = task_row.as_dict()
            samples, resample_attempts = generate_mixed_group(model, tokenizer, task, nominal_step, rejection_path, live, max_resample_attempts=max_resample_attempts)
            total_completions += resample_attempts * GROUP_SIZE
            if samples is None:
                metric = build_training_metric(nominal_step=nominal_step, task=task_row, rewards=None, kl=None, entropy=None,
                    mean_completion_length=None, optimizer_applied=False, skip_reason='resample_exhausted')
                metric['resample_attempts'] = resample_attempts
                metric['total_completions'] = total_completions
                append_jsonl(metrics_path, metric)
                skip_counts['resample_exhausted'] += 1
                continue
            reward_values = [float(s['training_reward']) for s in samples]
            mean_length = fmean(s['completion_tokens'] for s in samples)
            advantages = torch.tensor(compute_rloo_advantages(reward_values), device='cuda')
            step_stats = backward_rloo_policy_step(
                optimizer, model,
                advantages=advantages,
                samples=samples,
                beta=beta,
                token_statistics_fn=token_statistics,
            )
            optimizer_applied_steps += 1
            metric = build_training_metric(nominal_step=nominal_step, task=task_row, rewards=reward_values,
                kl=step_stats['kl'], entropy=step_stats['entropy'],
                mean_completion_length=mean_length, optimizer_applied=True, skip_reason=None, group_rewards=reward_values)
            metric.update({
                'loss': step_stats['loss'], 'mean_eval_reward': fmean(float(s['eval_reward'] or 0) for s in samples),
                'training_rubric_version': TRAINING_RUBRIC_VERSION_BINARY, 'eval_rubric_version': RUBRIC_VERSION,
                'resample_attempts': resample_attempts, 'total_completions': total_completions,
                'advantage_estimator': 'rloo',
            })
            append_jsonl(metrics_path, metric)
            for sample in samples:
                append_jsonl(rollout_path, {k: v for k, v in sample.items() if k not in ('prompt_ids', 'completion_ids')})
            mode = checkpoint_mode_v3(nominal_step=nominal_step, optimizer_applied_steps=optimizer_applied_steps,
                total_steps=target_optimizer_steps, smoke_resume=smoke_resume)
            if mode == 'full':
                save_checkpoint(run_dir, f'checkpoint-opt{optimizer_applied_steps}', model, optimizer, optimizer_applied_steps, mode='full')
            if smoke_resume and optimizer_applied_steps == target_optimizer_steps:
                save_checkpoint(run_dir, f'checkpoint-opt{optimizer_applied_steps}', model, optimizer, optimizer_applied_steps, mode='full')
            elif mode == 'adapter':
                save_checkpoint(run_dir, f'checkpoint-opt{optimizer_applied_steps}-adapter', model, mode='adapter')
        (run_dir / 'budget.json').write_text(json.dumps({**budget_status(total_completions), 'nominal_steps': nominal_step}, indent=2))
    model.save_pretrained(run_dir / 'final-adapter')
    tokenizer.save_pretrained(run_dir / 'final-adapter')
    return model, tokenizer, run_dir



In [ ]:
import sys
from pathlib import Path

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

if RESUME_RL_ONLY:
    print('Skipping base held-out — loaded from Drive')
else:
    sft_model, sft_tokenizer = build_policy()
    BASE_HELDOUT = evaluate_policy(sft_model, sft_tokenizer, 'base')
    print('base held-out:', json.dumps(BASE_HELDOUT, indent=2))



In [ ]:
import sys
from pathlib import Path

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

if RESUME_RL_ONLY:
    print('Skipping SFT — adapter on Drive:', sft_adapter_dir(OUTPUT_ROOT))
    assert sft_adapter_ready(OUTPUT_ROOT), 'Resume requires SFT adapter on Drive'
else:
    run_sft(sft_model, sft_tokenizer, epochs=1, max_examples=min(100, len(TRAIN_TASK_ROWS)))
    POST_SFT_HELDOUT = evaluate_policy(sft_model, sft_tokenizer, 'post_sft')
    GATE = evaluate_sft_gate(POST_SFT_HELDOUT)
    (OUTPUT_ROOT / 'gate_result.json').write_text(json.dumps(GATE, indent=2))
    print('SFT gate:', json.dumps(GATE, indent=2))
    if not GATE['passed']:
        raise SystemExit('SFT gate failed')
    assert sft_adapter_ready(OUTPUT_ROOT), 'SFT adapter missing after run_sft'
    print('SFT adapter saved:', sft_adapter_dir(OUTPUT_ROOT))



In [ ]:
import sys
from pathlib import Path

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

if RESUME_RL_ONLY:
    print('Skipping preflight — loaded from Drive')
else:
    PREFLIGHT = run_preflight(sft_model, sft_tokenizer)
    try:
        validate_trainable_pool_v3(PREFLIGHT)
    except RuntimeError as exc:
        print('Preflight pool warning:', exc)
    TRAINABLE_IDS = set(PREFLIGHT['trainable_task_ids'])
    if len(TRAINABLE_IDS) < 3:
        raise RuntimeError('Insufficient trainable tasks after preflight')
    TRAINING_SCHEDULE = build_training_schedule_v3(TRAINABLE_IDS, TRAIN_TASK_ROWS, TARGET_OPTIMIZER_STEPS)
    (OUTPUT_ROOT / 'preflight.json').write_text(json.dumps(PREFLIGHT, indent=2))
    print('Trainable:', PREFLIGHT['trainable_by_tier'])

log_gpu_memory('after preflight — keeping sft_model for smoke reuse')



In [ ]:
import sys
from pathlib import Path
import shutil

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

if not GATE['passed']:
    raise SystemExit('SFT gate failed')

smoke_dir = OUTPUT_ROOT / 'smoke'
if smoke_dir.exists() and not any(smoke_dir.glob('checkpoint-opt*')):
    shutil.rmtree(smoke_dir)

if 'sft_model' not in globals():
    assert_gpu_headroom(label='smoke reload')
    sft_model, sft_tokenizer = build_policy()
    load_sft_adapter_weights(sft_model, sft_adapter_dir(OUTPUT_ROOT))
    log_gpu_memory('smoke loaded SFT adapter from Drive')

assert_gpu_headroom(label='smoke start')
smoke_model, smoke_tokenizer, smoke_dir = train_run(
    BETAS[0], 'smoke', SMOKE_STEPS, TRAINABLE_IDS,
    TRAINING_SCHEDULE[:SMOKE_STEPS], smoke_resume=True,
    policy_model=sft_model, policy_tokenizer=sft_tokenizer,
    max_resample_attempts=SMOKE_MAX_RESAMPLE_ATTEMPTS,
)
ckpt = sorted(smoke_dir.glob('checkpoint-opt*-adapter')) or sorted(smoke_dir.glob('checkpoint-opt*'))
assert ckpt, 'Smoke produced no checkpoint'
full_ckpt = next((p for p in smoke_dir.glob('checkpoint-opt*') if (p / 'state.pt').exists()), None)
if full_ckpt:
    smoke_model, smoke_tokenizer, smoke_dir = train_run(
        BETAS[0], 'smoke', SMOKE_STEPS + 1, TRAINABLE_IDS,
        TRAINING_SCHEDULE[:SMOKE_STEPS + 1], resume_checkpoint=full_ckpt, smoke_resume=True,
        policy_model=smoke_model, policy_tokenizer=smoke_tokenizer,
        max_resample_attempts=SMOKE_MAX_RESAMPLE_ATTEMPTS,
    )
print('Smoke passed')
log_gpu_memory('after smoke')
hard_release_gpu_memory(globals(), 'smoke_model', 'smoke_tokenizer')
log_gpu_memory('after smoke release')



In [ ]:
import sys
from pathlib import Path

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

if not GATE['passed']:
    raise SystemExit('SFT gate failed')

assert_gpu_headroom(label='beta-0.01 start')
low_model, low_tokenizer, low_dir = train_run(BETAS[0], 'beta-0.01', TARGET_OPTIMIZER_STEPS, TRAINABLE_IDS, TRAINING_SCHEDULE)
LOW_HELDOUT = evaluate_policy(low_model, low_tokenizer, 'beta-0.01')
hard_release_gpu_memory(globals(), 'low_model', 'low_tokenizer')
log_gpu_memory('after beta-0.01 release')

assert_gpu_headroom(label='beta-0.1 start')
strong_model, strong_tokenizer, strong_dir = train_run(BETAS[1], 'beta-0.1', TARGET_OPTIMIZER_STEPS, TRAINABLE_IDS, TRAINING_SCHEDULE)
STRONG_HELDOUT = evaluate_policy(strong_model, strong_tokenizer, 'beta-0.1')
hard_release_gpu_memory(globals(), 'strong_model', 'strong_tokenizer')

COMPARISON = {
    'base': BASE_HELDOUT,
    'post_sft': POST_SFT_HELDOUT,
    'beta-0.01': LOW_HELDOUT,
    'beta-0.1': STRONG_HELDOUT,
    'training_rubric_version': TRAINING_RUBRIC_VERSION_BINARY,
    'eval_rubric_version': RUBRIC_VERSION,
    'gate': GATE,
}
(OUTPUT_ROOT / 'heldout_comparison.json').write_text(json.dumps(COMPARISON, indent=2))
print(json.dumps(COMPARISON, indent=2))



In [ ]:
import sys
from pathlib import Path

REPO = Path('/content/evaluator_gym') if Path('/content/evaluator_gym').exists() else (Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent)
_src = str((REPO / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
from evaluator_gym.training.colab_bootstrap import ensure_colab_repo_path
ensure_colab_repo_path(REPO)

import matplotlib.pyplot as plt

if GATE['passed']:
    FIGURE_DIR = OUTPUT_ROOT / 'figures'
    FIGURE_DIR.mkdir(exist_ok=True)
    runs = {'beta=0.01': read_jsonl(low_dir / 'metrics.jsonl'), 'beta=0.1': read_jsonl(strong_dir / 'metrics.jsonl')}

    def save_curve(field, ylabel, filename):
        figure, axis = plt.subplots(figsize=(7, 4))
        for label, rows in runs.items():
            points = [(row['nominal_step'], row[field]) for row in rows if row.get(field) is not None and row.get('optimizer_applied')]
            axis.plot([x for x, _ in points], [y for _, y in points], marker='o', markersize=3, label=label)
        axis.set(xlabel='Nominal step', ylabel=ylabel, title=ylabel)
        axis.grid(alpha=0.25)
        axis.legend()
        figure.savefig(FIGURE_DIR / f'{filename}.png', bbox_inches='tight')
        plt.show()

    save_curve('mean_reward', 'Mean binary training reward', 'reward')
    save_curve('kl', 'Frozen-reference k3 KL', 'kl')
    save_curve('total_completions', 'Cumulative completions', 'budget')
    save_curve('resample_attempts', 'Resample attempts per step', 'resample')

    labels = ['base', 'post_sft', 'beta-0.01', 'beta-0.1']
    means = [COMPARISON[label]['mean_reward_scored'] or 0 for label in labels]
    figure, axis = plt.subplots(figsize=(8, 4))
    axis.bar(labels, means)
    axis.set(title='Held-out eval rubric (0.1.2)', ylabel='Mean eval reward', ylim=(0, 1))
    figure.savefig(FIGURE_DIR / 'heldout-before-after.png', bbox_inches='tight')
    plt.show()

    AUDIT = {'beta-0.01': exploit_search_v3(low_dir), 'beta-0.1': exploit_search_v3(strong_dir)}
    (OUTPUT_ROOT / 'exploit_search.json').write_text(json.dumps(AUDIT, indent=2))
    print(json.dumps(AUDIT, indent=2))

print('After the run, copy artifacts to repo for dashboard:')
print(' ', RESULTS_STAGING_ROOT_V3)
